# Libaray Load

In [1]:
import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
from pyproj import Proj, Transformer
from scipy.spatial import cKDTree

# 관측소 데이터 가져오기

In [2]:
aa=pd.DataFrame()
for node in range(0,6):
    # 요청 변수
    params = {
        "basin": f"{node}",       # 한강
        "oper": "y",        # 운영중
      #  "mngorg": "1",      # 기후에너지환경부
        "output": "json"    # JSON 형식
    }

    # API URL
    url = "http://www.wamis.go.kr:8080/wamis/openapi/wkw/wl_dubwlobs"

    # 요청
    response = requests.get(url, params=params)

    # 상태 확인
    if response.status_code == 200:
        data = response.json()  # JSON 파싱
        obs_list = data.get("list", [])  # 관측소 리스트
        df = pd.DataFrame(obs_list)
        print(df.head())
    else:
        print("API 요청 실패:", response.status_code)
    aa=pd.concat([aa,df],axis=0)

Empty DataFrame
Columns: []
Index: []
  bbsnnm    obscd     obsnm clsyn obsknd  sbsncd    mngorg
0     한강  1001602  평창군(송정교)    운영    T/M  100108  기후에너지환경부
1     한강  1001603  태백시(무사교)    운영    T/M  100101   한국수자원공사
2     한강  1001605      송천1교    운영    T/M  100105   한국수력원자력
3     한강  1001607  삼척시(번천교)    운영    T/M  100101   한국수자원공사
4     한강  1001613  삼척시(광동교)    운영    T/M  100102   한국수자원공사
  bbsnnm    obscd      obsnm clsyn obsknd  sbsncd    mngorg
0    낙동강  2001608  태백시(보드미교)    운영    T/M  200101  기후에너지환경부
1    낙동강  2001610   태백시(문화교)    운영    T/M  200101  기후에너지환경부
2    낙동강  2001613   태백시(루사교)    운영    T/M  200101   한국수자원공사
3    낙동강  2001615   봉화군(대현교)    운영    T/M  200102  기후에너지환경부
4    낙동강  2001625   봉화군(광회교)    운영    T/M  200103  기후에너지환경부
  bbsnnm    obscd       obsnm clsyn obsknd  sbsncd    mngorg
0     금강  3001605    장수군(운곡교)    운영    T/M  300101  기후에너지환경부
1     금강  3001610  장수군(장계시장교)    운영    T/M  300102  기후에너지환경부
2     금강  3001615          천천    운영     보통  300104   한국수자원공사
3   

In [3]:
aa.reset_index(drop=True,inplace=True)

In [4]:
aa

,bbsnnm,obscd,obsnm,clsyn,obsknd,sbsncd,mngorg
0,한강,1001602,평창군(송정교),운영,T/M,100108,기후에너지환경부
1,한강,1001603,태백시(무사교),운영,T/M,100101,한국수자원공사
2,한강,1001605,송천1교,운영,T/M,100105,한국수력원자력
3,한강,1001607,삼척시(번천교),운영,T/M,100101,한국수자원공사
4,한강,1001613,삼척시(광동교),운영,T/M,100102,한국수자원공사
...,...,...,...,...,...,...,...
1182,영산강서해,5301660,고창군(부정교),운영,T/M,530103,기후에너지환경부
1183,영산강서해,5301690,고창군(용선교),운영,T/M,530103,기후에너지환경부
1184,영산강서해,5302620,영광군(반와교),운영,T/M,530203,기후에너지환경부
1185,영산강서해,5302640,불갑,운영,T/M,530204,한국농어촌공사


In [5]:
vv=pd.DataFrame()
for node in range(0,1187):
    num=aa['obscd'][node]
    # 요청 변수
    params = {
        "obscd": num,       # 한강
        "output": "json"    # JSON 형식
    }

    # API URL
    url = "http://www.wamis.go.kr:8080/wamis/openapi/wkw/wl_obsinfo"

    # 요청
    response = requests.get(url, params=params)

    # 상태 확인
    if response.status_code == 200:
        data = response.json()  # JSON 파싱
        obs_list = data.get("list", [])  # 관측소 리스트
        df = pd.DataFrame(obs_list)
        print(df.head())
    else:
        print("API 요청 실패:", response.status_code)
    vv=pd.concat([vv,df],axis=0)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
      obsnm             obsnmeng  wlobscd    mggvcd bbsncd  sbsncd  \
0  울산시(웅촌교)  Ulsansi(Ungchongyo)  2301625  기후에너지환경부  회야ㆍ수영  230102   

     obsopndt obskdcd rivnm bsnara  ...     tmx     tmy     gdt   wltel tdeyn  \
0  2025-01-01     T/M   곡천천  19.38  ...  400230  220088  49.419  60.156     N   

  mxgrd sistartobsdh  siendobsdh olstartobsdh olendobsdh  
0  None   2025010100  2025101622     20241231   20251016  

[1 rows x 27 columns]
      obsnm               obsnmeng  wlobscd    mggvcd bbsncd  sbsncd  \
0  울산시(통천교)  Ulsansi(Tongcheongyo)  2301630  기후에너지환경부  회야ㆍ수영  230102   

     obsopndt obskdcd rivnm bsnara  ...     tmx     tmy    gdt  wltel tdeyn  \
0  1962-07-01     T/M   회야강  112.8  ...  403251  221544  33.72  38.95     N   

  mxgrd sistartobsdh  siendobsdh olstartobsdh olendobsdh  
0     5   2006010916  2025101622     19630101   20251016  

[1 rows x 27 columns]
      obsnm                obsnmeng  wlobscd    mggvcd bbsncd  sbsncd  \
0

In [6]:
vv.reset_index(drop=True,inplace=True)

In [7]:
got=vv[['obsnm','wlobscd','tmx','tmy']]
got.dropna(axis=0,inplace=True)

/tmp/ipython-input-1119914289.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  got.dropna(axis=0,inplace=True)


# 관측소 좌표변환

In [72]:
got

,obsnm,wlobscd,tmx,tmy,lon,lat
0,평창군(송정교),1001602,336935,459417,120.992317,23.960876
1,태백시(무사교),1001603,376803,425838,121.395987,23.675392
2,송천1교,1001605,350849,464532,121.125942,24.012515
3,삼척시(번천교),1001607,376860,431330,121.394240,23.724738
4,삼척시(광동교),1001613,372665,430594,121.353631,23.716502
...,...,...,...,...,...,...
1182,고창군(부정교),5301660,165839,219581,119.452898,21.735976
1183,고창군(용선교),5301690,163268,225910,119.425095,21.791382
1184,영광군(반와교),5302620,155189,202308,119.359672,21.576478
1185,불갑,5302640,155265,192198,119.365470,21.486094


In [89]:
tm_proj = Proj(init='epsg:2097')   # TM 좌표
wgs84_proj = Proj(proj='latlong', datum='WGS84')  # 위경도

# 변환기 생성
transformer = Transformer.from_proj(tm_proj, wgs84_proj)

# 변환 수행
got['lon'], got['lat'] = transformer.transform(got['tmx'].values, got['tmy'].values)

/usr/local/lib/python3.12/dist-packages/pyproj/crs/crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
/tmp/ipython-input-2254988868.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  got['lon'], got['lat'] = transformer.transform(got['tmx'].values, got['tmy'].values)


In [90]:
adj = pd.read_csv('/content/drive/MyDrive/지하수예측경진대회/12개_지하수_지점_정보.csv')

In [91]:
adj=adj[['code_new','위도','경도','수리전도도(cm/sec)','표고(EL.m)']]

In [92]:
obs=got[['wlobscd','lon','lat']]

In [93]:
# 시작과 종료 날짜
start_date = pd.to_datetime('20140101')
end_date = pd.to_datetime('20231231')

start_list = []
end_list = []

current_start = start_date

while current_start <= end_date:
    current_end = current_start + pd.DateOffset(months=6) - pd.Timedelta(days=1)
    if current_end > end_date:
        current_end = end_date

    start_list.append(current_start.strftime('%Y%m%d'))
    end_list.append(current_end.strftime('%Y%m%d'))

    current_start = current_end + pd.Timedelta(days=1)

In [94]:
eleven = []

for node in tqdm(range(0, 12)):
    for k in tqdm(range(1, 20)):  # k번째로 가까운 관측소
        obs_coords = obs[['lat', 'lon']].to_numpy()
        tree = cKDTree(obs_coords)

        # adj 좌표
        adj_coords = adj.loc[node, ['위도', '경도']].to_numpy()

        # k번째로 가까운 관측소 찾기
        distances, indices = tree.query(adj_coords, k=k)
        if k == 1:
            second_distances = distances
            second_indices = indices
        else:
            second_distances = distances[k - 1]
            second_indices = indices[k - 1]

        # adj에 wlobscd, distance 추가
        adj.loc[node, 'wlobscd'] = obs.iloc[second_indices]['wlobscd']
        adj.loc[node, 'distance'] = second_distances

        node_dfs = []
        data_found = False

        for time in range(len(start_list)):
            params = {
                "obscd": adj['wlobscd'][node],
                "startdt": start_list[time],
                "enddt": end_list[time],
                "output": "json"
            }

            url = 'http://www.wamis.go.kr:8080/wamis/openapi/wkw/wl_hrdata'
            response = requests.get(url, params=params)

            if response.status_code == 200:
                data = response.json()
                obs_list = data.get("list", [])

                if len(obs_list) > 0:
                    df = pd.DataFrame(obs_list)
                    node_dfs.append(df)
            else:
                print(f"API 요청 실패: {response.status_code}")

        # time별 DataFrame 합치기
        if len(node_dfs) > 0:
            node_df = pd.concat(node_dfs, axis=0, ignore_index=True, sort=False)

            # 길이가 80000 이상이면 append, 아니면 다음 k
            if len(node_df) > 80000:
                eleven.append(node_df)
                break  # 조건 만족했으므로 k 루프 종료
            else:
                # 다음 k 시도
                continue

 92%|█████████▏| 11/12 [05:59<00:32, 32.69s/it]


KeyboardInterrupt: 

In [96]:
adj=pd.merge(got[['obsnm','lon','lat','wlobscd']],adj,on='wlobscd',how='right')

In [99]:
adj.to_csv('/content/drive/MyDrive/지하수예측경진대회/12개_지하수_지점_정보.csv',index=False)

In [104]:
eleven[0]

,ymdh,wl,code_new
0,2014010101,.35,1
1,2014010102,.35,1
2,2014010103,.35,1
3,2014010104,.35,1
4,2014010105,.35,1
...,...,...,...
87516,2023123120,1.27,1
87517,2023123121,1.27,1
87518,2023123122,1.27,1
87519,2023123123,1.27,1


In [106]:
for node in range(12):
# 리스트의 DataFrame을 하나로 합치기
# bb 안에 리스트가 들어있는 경우 flatten
    final_df = pd.concat(eleven, axis=0, ignore_index=True)

In [107]:
final_df

,ymdh,wl,code_new
0,2014010101,.35,1
1,2014010102,.35,1
2,2014010103,.35,1
3,2014010104,.35,1
4,2014010105,.35,1
...,...,...,...
944651,2023123120,1.31,11
944652,2023123121,1.32,11
944653,2023123122,1.33,11
944654,2023123123,1.32,11


In [108]:
# ymdh가 2023123124인 행 제거
final_df = final_df[final_df['ymdh'] != '2023123124']

In [109]:
# 문자열 길이 10자리만 사용
final_df['ymdh_fixed'] = final_df['ymdh'].str[:10]
# datetime 변환
final_df['ymd'] = pd.to_datetime(final_df['ymdh_fixed'], format='%Y%m%d%H', errors='coerce')

/tmp/ipython-input-4047605523.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['ymdh_fixed'] = final_df['ymdh'].str[:10]
/tmp/ipython-input-4047605523.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['ymd'] = pd.to_datetime(final_df['ymdh_fixed'], format='%Y%m%d%H', errors='coerce')


In [110]:
final_df.drop(['ymdh_fixed','ymdh'],axis=1,inplace=True)

/tmp/ipython-input-3417081849.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.drop(['ymdh_fixed','ymdh'],axis=1,inplace=True)


In [111]:
df_no_dup = final_df.sort_values(by=['code_new','ymd'],ascending=True)

In [112]:
df = df_no_dup[df_no_dup['ymd'].notna()]

In [114]:
df.to_csv('/content/drive/MyDrive/지하수예측경진대회/근처하천수위.csv',index=False)